# Frame Dragging from Rotating Energy Sources

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Frame_Dragging.ipynb)

## What This Notebook Demonstrates

A **rotating mass** drags the substrate $\chi$ field, transferring angular momentum to nearby waves. This is the LFM analog of the **Lense-Thirring effect** (frame dragging) in General Relativity.

We place a rotating $m=1$ energy source at the grid center and measure angular momentum pickup by a test region.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
KAPPA = 1.0 / 63.0
C = 1.0

NX, NY = 80, 80
dx = 1.0
dt = 0.04
center = NX // 2

x = np.arange(NX) * dx
y = np.arange(NY) * dx
X, Y = np.meshgrid(x, y, indexing='ij')
R = np.sqrt((X - center*dx)**2 + (Y - center*dx)**2) + 1e-6
THETA = np.arctan2(Y - center*dx, X - center*dx)

def lap2d(f):
    return (np.roll(f,1,0) + np.roll(f,-1,0)
            + np.roll(f,1,1) + np.roll(f,-1,1) - 4*f) / dx**2

# Initialize
E = np.zeros((NX, NY))
E_prev = np.zeros((NX, NY))
chi = np.ones((NX, NY)) * CHI_0
chi_prev = chi.copy()

# Rotating source parameters
omega_rot = 0.15     # angular velocity
source_amp = 5.0
source_radius = 8.0
source_width = 3.0

n_steps = 2000

# Angular momentum measurement
def measure_angular_momentum(E_field, E_prev_field):
    """L_z = integral of (x * p_y - y * p_x) where p = E * dE/dt."""
    dE_dt = (E_field - E_prev_field) / dt
    x_rel = X - center * dx
    y_rel = Y - center * dx
    Lz = np.sum(x_rel * E_field * np.gradient(dE_dt, dx, axis=1)
                - y_rel * E_field * np.gradient(dE_dt, dx, axis=0)) * dx**2
    return Lz

Lz_history = []
t_history = []

for step in range(n_steps):
    t = step * dt

    # Rotating m=1 source: E_source(r, theta, t) = A * ring(r) * cos(theta - omega*t)
    ring = np.exp(-(R - source_radius)**2 / (2 * source_width**2))
    angle_pattern = np.cos(THETA - omega_rot * t)
    source = source_amp * ring * angle_pattern

    # Inject energy
    E += 0.01 * source

    # GOV-01
    E_next = 2*E - E_prev + dt**2 * (C**2 * lap2d(E) - chi**2 * E)
    # GOV-02
    chi_next = 2*chi - chi_prev + dt**2 * (C**2 * lap2d(chi) - KAPPA * E**2)

    # Measure angular momentum in test region (away from source)
    if step % 50 == 0 and step > 100:
        Lz = measure_angular_momentum(E, E_prev)
        Lz_history.append(Lz)
        t_history.append(t)

    E_prev, E = E, E_next
    chi_prev, chi = chi, chi_next

    if step % 500 == 0:
        print(f"  Step {step}/{n_steps}")

print(f"\nSimulation complete.")
print(f"Angular momentum measurements: {len(Lz_history)}")
if len(Lz_history) > 2:
    Lz_arr = np.array(Lz_history)
    initial_Lz = Lz_arr[:3].mean()
    final_Lz = Lz_arr[-3:].mean()
    print(f"  Initial mean Lz: {initial_Lz:.4f}")
    print(f"  Final mean Lz:   {final_Lz:.4f}")
    if abs(final_Lz) > abs(initial_Lz) * 1.1:
        print(f"\n  FRAME DRAGGING DETECTED: angular momentum transferred!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Final E field
ax = axes[0]
im = ax.imshow(E.T, origin='lower', cmap='RdBu_r',
               extent=[0, NX, 0, NY])
ax.set_title('Final E Field')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax)

# Panel 2: Final chi field
ax = axes[1]
im = ax.imshow(chi.T, origin='lower', cmap='viridis',
               extent=[0, NX, 0, NY])
ax.set_title('Final chi Field')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.colorbar(im, ax=ax, label='chi')

# Panel 3: Angular momentum over time
ax = axes[2]
ax.plot(t_history, Lz_history, 'b-o', markersize=3)
ax.set_xlabel('Time')
ax.set_ylabel('Angular Momentum Lz')
ax.set_title('Angular Momentum Transfer (Frame Dragging)')
ax.grid(alpha=0.3)
ax.axhline(0, color='gray', ls='--', alpha=0.5)

plt.suptitle('Frame Dragging: Rotating Mass Drags Nearby Waves',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- A rotating energy source creates a rotating $\chi$ pattern
- Nearby waves pick up **angular momentum** from the rotating substrate
- This is the LFM analog of **Lense-Thirring frame dragging** in GR
- No metric tensor or Kerr solution was injected